# Export HILDA+ for Lithuania (PNG + GeoTIFF + CSV)

**Kernel:** use an environment with `xarray`, `shapely`, `numpy`, `pandas`, `matplotlib`, and (optional) `rasterio` for GeoTIFFs.

**Working directory:** run from anywhere inside the `Data` repo; the next cell finds the root (folder that contains `lt_subbasins.json`).

**Outputs:**
- `rasters/hilda/hilda_YYYY.png`
- `rasters/hilda/geotiff/hilda_YYYY.tif` (if rasterio/PROJ work)
- `outputs/hilda_lithuania_timeseries.csv`

In [1]:
from pathlib import Path


def find_data_root(start: Path) -> Path:
    for d in [start, *start.parents]:
        if (d / "lt_subbasins.json").is_file():
            return d
    raise FileNotFoundError(
        "Could not find lt_subbasins.json. Open this notebook from your Data folder, "
        "or set DATA_ROOT below manually."
    )


DATA_ROOT = find_data_root(Path.cwd())
# DATA_ROOT = Path(r"C:\Users\YOU\Desktop\LEI\Data")  # uncomment to force

LT_GEOJSON = DATA_ROOT / "lt_boundary_admin.json"
HILDA_NC = (
    DATA_ROOT
    / "Hilda"
    / "Version 2.0"
    / "Winkler-etal_2025_allfiles"
    / "hildap_vGLOB-2.0_netCDF_extended-time"
    / "hildaplus_GLOB-2-0_states.nc"
)
OUT_RASTERS = DATA_ROOT / "rasters" / "hilda"
OUT_GEOTIFF = OUT_RASTERS / "geotiff"
OUT_CSV = DATA_ROOT / "outputs" / "hilda_lithuania_timeseries.csv"

OUT_RASTERS.mkdir(parents=True, exist_ok=True)
OUT_GEOTIFF.mkdir(parents=True, exist_ok=True)
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT.resolve())
print("HILDA_NC exists:", HILDA_NC.is_file())

DATA_ROOT: C:\Users\matas\Desktop\LEI\Data
HILDA_NC exists: True


In [2]:
import json
from shapely.geometry import shape, Point
from shapely.ops import unary_union
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

with open(LT_GEOJSON, "r", encoding="utf-8") as f:
    lt_data = json.load(f)
geoms = [shape(feat["geometry"]) for feat in lt_data["features"]]
lt_geom = unary_union(geoms)


def polygon_mask(lat_vals, lon_vals, geom):
    h, w = len(lat_vals), len(lon_vals)
    mask = np.zeros((h, w), dtype=bool)
    for i, lat in enumerate(lat_vals):
        for j, lon in enumerate(lon_vals):
            mask[i, j] = geom.contains(Point(lon, lat))
    return mask


LAT_MIN, LAT_MAX = 53.5, 56.6
LON_MIN, LON_MAX = 20.5, 26.7

print("Opening:", HILDA_NC)
ds = xr.open_dataset(HILDA_NC, chunks="auto")
da = ds["LULC_states"]

lat_name, lon_name = "latitude", "longitude"
lats = da[lat_name].values
lat_slice = (
    slice(LAT_MAX, LAT_MIN)
    if (len(lats) > 1 and lats[0] > lats[-1])
    else slice(LAT_MIN, LAT_MAX)
)
da_lt = da.sel(**{lat_name: lat_slice, lon_name: slice(LON_MIN, LON_MAX)})

lat_vals = da_lt[lat_name].values
lon_vals = da_lt[lon_name].values
mask = polygon_mask(lat_vals, lon_vals, lt_geom)
print("Subset shape:", da_lt.shape)

Opening: C:\Users\matas\Desktop\LEI\Data\Hilda\Version 2.0\Winkler-etal_2025_allfiles\hildap_vGLOB-2.0_netCDF_extended-time\hildaplus_GLOB-2-0_states.nc
Subset shape: (121, 310, 620)


In [3]:
groups = {
    "Water": [0, 77],
    "Urban": [11],
    "Agriculture": [22, 23, 24, 33],
    "Forest": [41, 42, 43, 44, 45],
}

five_map = {}
for c in groups["Water"]:
    five_map[c] = 1
for c in groups["Urban"]:
    five_map[c] = 3
for c in groups["Agriculture"]:
    five_map[c] = 4
for c in groups["Forest"]:
    five_map[c] = 5

class_names = {1: "Water", 3: "Urban", 4: "Agriculture", 5: "Forest"}
colors = ["#4DA6FF", "#7B68EE", "#FF4D4D", "#FFD24D", "#228B22"]
cmap = ListedColormap(colors)
cmap.set_bad((0, 0, 0, 0))
norm = matplotlib.colors.Normalize(vmin=1, vmax=5)


def reclass_to_five(layer):
    data = layer.values
    out = np.full_like(data, fill_value=np.nan, dtype="float32")
    for orig, unified in five_map.items():
        out[data == orig] = unified
    return out


time_vals = da_lt["time"].values
year_to_ti = {}
for ti, t_val in enumerate(time_vals):
    y = int(round(float(t_val)))
    if 1910 <= y <= 2020:
        year_to_ti[y] = ti
years_export = np.array(sorted(year_to_ti.keys()), dtype=int)
print("Years to export:", len(years_export), "—", years_export[:5], "...", years_export[-3:])

Years to export: 111 — [1910 1911 1912 1913 1914] ... [2018 2019 2020]


In [4]:
try:
    import rasterio
    from rasterio.transform import from_bounds as _from_bounds
except ImportError:
    rasterio = None
    _from_bounds = None

# Only clear old GeoTIFFs when rasterio imports; otherwise keep existing .tif (CSV/PNG still update).
if rasterio is not None:
    for _p in OUT_GEOTIFF.glob("hilda_*.tif"):
        try:
            _p.unlink()
        except OSError:
            pass
else:
    print(
        "rasterio not importable — not clearing geotiff/*.tif; conda install -c conda-forge rasterio gdal"
    )

records = []
for year in years_export:
    layer = da_lt.isel(time=year_to_ti[int(year)])
    arr = reclass_to_five(layer)
    arr_masked = np.where(mask, arr, np.nan)

    flat = arr_masked[np.isfinite(arr_masked)].astype(int)
    if flat.size:
        uniq, cnts = np.unique(flat, return_counts=True)
        for cls_id, cnt in zip(uniq, cnts, strict=False):
            records.append(
                (int(year), int(cls_id), class_names.get(int(cls_id), f"class_{cls_id}"), int(cnt))
            )

    rgba = cmap(norm(arr_masked))
    out_png = OUT_RASTERS / f"hilda_{int(year)}.png"
    plt.imsave(out_png, rgba)

    if rasterio is not None:
        try:
            h, w = arr_masked.shape
            west, east = float(np.min(lon_vals)), float(np.max(lon_vals))
            south, north = float(np.min(lat_vals)), float(np.max(lat_vals))
            arr_uint8 = np.where(np.isfinite(arr_masked), arr_masked.astype(np.uint8), 0)
            transform = _from_bounds(west, south, east, north, w, h)
            out_tif = OUT_GEOTIFF / f"hilda_{int(year)}.tif"
            profile = dict(
                driver="GTiff",
                height=h,
                width=w,
                count=1,
                dtype=arr_uint8.dtype,
                transform=transform,
                nodata=0,
            )
            last_err = None
            for crs_arg in ("EPSG:4326", None):
                try:
                    kw = {**profile, **({} if crs_arg is None else {"crs": crs_arg})}
                    with rasterio.open(out_tif, "w", **kw) as dst:
                        dst.write(arr_uint8, 1)
                    break
                except Exception as e:
                    last_err = e
            else:
                assert last_err is not None
                raise last_err
            if int(year) % 20 == 0 or year == years_export[-1]:
                print("Saved", out_tif.name)
        except Exception as e:
            print(f"GeoTIFF skipped {year}:", e)

df = pd.DataFrame(records, columns=["year", "class_id", "class_name", "count"])
df.to_csv(OUT_CSV, index=False)
print("Done. CSV:", OUT_CSV)
print(df.head())

C:\Users\matas\AppData\Local\Temp\ipykernel_31796\367737717.py:26: RuntimeWarning: invalid value encountered in cast
  arr_uint8 = np.where(np.isfinite(arr_masked), arr_masked.astype(np.uint8), 0)


Saved hilda_1920.tif
Saved hilda_1940.tif
Saved hilda_1960.tif
Saved hilda_1980.tif
Saved hilda_2000.tif
Saved hilda_2020.tif
Done. CSV: C:\Users\matas\Desktop\LEI\Data\outputs\hilda_lithuania_timeseries.csv
   year  class_id   class_name  count
0  1910         1        Water    988
1  1910         3        Urban    355
2  1910         4  Agriculture  61632
3  1910         5       Forest  27784
4  1911         1        Water    988


### Single source of truth (recommended)

The logic lives in `analysis/export_hilda_lithuania.py` (same folder as `notebooks/`). Run it from the repo root so PNGs, GeoTIFFs, and `outputs/hilda_lithuania_timeseries.csv` stay in sync.

```bash
python analysis/export_hilda_lithuania.py
```

Or run the next cell (kernel cwd should be `analysis/notebooks/` or the Data repo root).

In [ ]:
# Run full export (NetCDF → PNG + GeoTIFF + CSV). Requires xarray, shapely, rasterio, etc.
%run ../export_hilda_lithuania.py